# Lab 11: Grid Localization using Bayes Filter (Virtual Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

<hr>


In [1]:
%load_ext autoreload
%autoreload 2

import traceback
from notebook_utils import *
from Traj import *
import asyncio
from localization_extras import Localization

# Setup Logger
LOG = get_logger('demo_notebook.log')

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander

gui.show()

# Start the simulator
START_SIM()

# Start the plotter
START_PLOTTER()


2026-04-27 16:24:33,117 | INFO     |: Logger demo_notebook.log initialized.


TwoByTwoLayout(children=(Label(value='Simulator', layout=Layout(grid_area='top-left', width='80px')), HBox(chi…

Loading Flatland...
Initializing pygame framework...


In [2]:
# Initialize Robot to communicate with the virtual robot and plotter
robot = VirtualRobot(cmdr)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a VirtualRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()


2026-04-27 16:24:34,402 | INFO     |:  | Number of observations per grid cell: 18
2026-04-27 16:24:34,402 | INFO     |:  | Precaching Views...


/Users/samb/Programming/fast-robots/labs/lab11/sim/localization.py:151: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(distance_intersections_tt), intersections_tt[np.nanargmin(distance_intersections_tt)]


2026-04-27 16:24:35,521 | INFO     |:  | Precaching Time: 1.118 secs
2026-04-27 16:24:35,522 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-27 16:24:35,523 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


# Run the Bayes Filter
The cells below utilizes the member functions of class **Localization** (defined in [localization_extras.py](../localization_extras.py)) in each iteration of the Bayes filter algorithm to localize the robot in the grid map. <br>

In [3]:
# Reset Robot and Plots
robot.reset()
cmdr.reset_plotter()

# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
await loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.print_update_stats(plot_data=True)

# Plot Odom and GT
current_odom, current_gt = robot.get_pose()
cmdr.plot_gt(current_gt[0], current_gt[1])
cmdr.plot_odom(current_odom[0], current_odom[1])


2026-04-27 16:24:35,951 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-27 16:24:35,951 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-04-27 16:24:38,902 | INFO     |: Update Step
2026-04-27 16:24:38,904 | INFO     |:      | Update Time: 0.001 secs
2026-04-27 16:24:38,904 | INFO     |: ---------- UPDATE STATS -----------
2026-04-27 16:24:38,910 | INFO     |: GT index      : (6, 4, 9)
2026-04-27 16:24:38,910 | INFO     |: Bel index     : (np.int64(5), np.int64(4), np.int64(9)) with prob = 0.9999980
2026-04-27 16:24:38,910 | INFO     |: Bel_bar prob at index = 0.00051440329218107
2026-04-27 16:24:38,911 | INFO     |: GT            : (0.000, 0.000, 360.000)
2026-04-27 16:24:38,911 | INFO     |: Belief        : (0.000, 0.000, 10.000)
2026-04-27 16:24:38,911 | INFO     |: POS ERROR     : (-0.000, -0.000, 350.000)
2026-04-27 16:24:38,912 | INFO     |: ---------- UPDATE STATS -----------


In [4]:
# Initialize the Trajectory object
traj = Trajectory(loc)

# Run through each motion steps
for t in range(0, traj.total_time_steps):
    print("\n\n-----------------", t, "-----------------")
    
    prev_odom, current_odom, prev_gt, current_gt = traj.execute_time_step(t)
        
    # Prediction Step
    loc.prediction_step(current_odom, prev_odom)
    loc.print_prediction_stats(plot_data=True)
    
    # Get Observation Data by executing a 360 degree rotation motion
    await loc.get_observation_data()
    
    # Update Step
    loc.update_step()
    loc.print_update_stats(plot_data=True)

# Uncomment the below line to wait for keyboard input between each iteration.
#   input("Press Enter to Continue")
        
    print("-------------------------------------")




----------------- 0 -----------------
2026-04-27 16:24:42,179 | INFO     |: Prediction Step
2026-04-27 16:24:42,198 | INFO     |:  | Prediction Time: 0.017 secs
2026-04-27 16:24:42,198 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-27 16:24:42,212 | INFO     |: GT index         : (6, 3, 6)
2026-04-27 16:24:42,213 | INFO     |: Prior Bel index  : (np.int64(4), np.int64(4), np.int64(7)) with prob = 0.1363584
2026-04-27 16:24:42,213 | INFO     |: POS ERROR        : (0.586, -0.089, -10.393)
2026-04-27 16:24:42,214 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-27 16:24:45,115 | INFO     |: Update Step
2026-04-27 16:24:45,116 | INFO     |:      | Update Time: 0.001 secs
2026-04-27 16:24:45,116 | INFO     |: ---------- UPDATE STATS -----------
2026-04-27 16:24:45,130 | INFO     |: GT index      : (6, 3, 6)
2026-04-27 16:24:45,130 | INFO     |: Bel index     : (np.int64(6), np.int64(4), np.int64(6)) with prob = 0.9999999
2026-04-27 16:24:45,131 | INFO     |: Be

2026-04-27 16:24:51.951 Python[36401:16948584] Warning: Window move completed without beginning


2026-04-27 16:24:54,498 | INFO     |: Update Step
2026-04-27 16:24:54,500 | INFO     |:      | Update Time: 0.001 secs
2026-04-27 16:24:54,500 | INFO     |: ---------- UPDATE STATS -----------
2026-04-27 16:24:54,502 | INFO     |: GT index      : (7, 2, 4)
2026-04-27 16:24:54,502 | INFO     |: Bel index     : (np.int64(7), np.int64(2), np.int64(4)) with prob = 0.9999999
2026-04-27 16:24:54,503 | INFO     |: Bel_bar prob at index = 3.5063139446301e-42
2026-04-27 16:24:54,503 | INFO     |: GT            : (0.503, -0.537, 993.386)
2026-04-27 16:24:54,503 | INFO     |: Belief        : (0.610, -0.610, -90.000)
2026-04-27 16:24:54,504 | INFO     |: POS ERROR     : (-0.107, 0.073, 1083.386)
2026-04-27 16:24:54,504 | INFO     |: ---------- UPDATE STATS -----------
-------------------------------------


----------------- 3 -----------------
2026-04-27 16:24:55,546 | INFO     |: Prediction Step
2026-04-27 16:24:55,561 | INFO     |:  | Prediction Time: 0.014 secs
2026-04-27 16:24:55,562 | INFO  